# Session 12, notebook 1 of 2: A team of specialists

**Week 6 · Session 2 · LangGraph III of III · Project version v9**

Session 11 gave the inbox a memory and a pause button. It is still one graph whose desks are fixed by
code, its policy desk knows one corpus and ignores the filters a question implies, and its tools exist
only inside one Python process. This notebook hires a team with a manager and teaches it to learn from how
past tickets were handled. Putting its tools on a protocol any AI host can use is the second notebook,
`session-12-mcp.ipynb`.

<img src="figures/session-map.png" width="900"
     alt="The whole of session 12 on one page. A top strip contrasts v8, one graph whose desks are fixed by
     code, with v9, a team with a manager whose tools also work outside the process. Below, three zones.
     Zone A, multi-agent honestly: when not to, five patterns, subagents as tools, the supervisor as a
     graph, subgraphs, handoffs, the measured comparison, skills. Zone B, the knowledge specialist: a
     second source, self-query filters, contextual and parent chunks, a cache, the decision table. Zone
     C, MCP: host, client and server, building the server, one server for many hosts, streamable HTTP,
     security. A bottom band says most tasks do not need a team, and the numbers decide.">

Session 12 has two notebooks. This one holds the two big topics; the map's third zone, MCP, is
`session-12-mcp.ipynb`:

| | Topic | Question it answers |
|---|---|---|
| **A** | **Multi-agent systems, honestly** | When does a team of agents beat one graph, and how do you prove it? |
| **B** | **The knowledge specialist** | How do you answer from two kinds of source, use the filters a question implies, and know a technique helped? |
| *C* | *MCP: the team's tools as a protocol* | *in `session-12-mcp.ipynb`* |

Part B stands on its own, so you can study it before Part A. By the end you will have most of **v9**: a
supervisor graph of specialists with visible routing, and a knowledge specialist that cites policies and
past tickets differently. The v9 snapshot already serves its read-only tools over MCP (the orders
specialist gets them that way); the MCP notebook opens that part up.

## 1. Setup

**Four new packages**, all in `pyproject.toml`, so after pulling run `uv sync` once:

| Package | What it is for |
|---|---|
| `mcp[cli]` | the official MCP Python SDK: our server (`FastMCP`) and `mcp dev`, which opens the MCP Inspector |
| `langchain-mcp-adapters` | the MCP **client** for LangChain 1.3: turns a server's tools into LangChain tools |
| `ddgs`, `langchain-tavily` | **[ADV]** web search for the Market Analyst |

**A version note you will meet online.** MCP's Python SDK 2.x renamed `FastMCP` to `MCPServer`, and
LangChain 1.4 ships its own client, `langchain.mcp.MCPAdapter`. This course is pinned to LangChain 1.3,
whose client (`langchain-mcp-adapters`) requires `mcp<2`, and one environment cannot hold both majors.
So the server here says `FastMCP`. For a server this size, little else differs.

**The v9 project changes more files than any session so far.** The notebook imports the real modules:

| Module | What it does |
|---|---|
| `core/` | **new home** of `config.py`, plus `constants.py` and `exceptions.py` (section 17) |
| `workflows/supervisor/` | **new**: the supervisor graph, its routing state, and `handoffs.py` |
| `workflows/specialists/` | **new**: orders (tools over MCP), knowledge (a subgraph), refunds, [ADV] market |
| `workflows/refund/` | **new**: v8's refund flow as its own graph |
| `workflows/subagents.py`, `workflows/skills.py` | **new**: pattern 1 (the team as one agent) and `load_skill` |
| `kb/` | past tickets as a second collection, self-query filters, contextual and parent schemes |
| `mcp_server/` | **new**: ShopWise's MCP server, a poisoned demo server, and a description audit (taught in `session-12-mcp.ipynb`) |
| `evals/` | precedent and filtered questions, and the multi-agent comparison |
| `chat.py`, `inbox.py` | one import changed each: they now open the team |

**Providers, as in session 11.** Answers come from `openrouter:openai/gpt-oss-20b`, embeddings from
Gemini's `gemini-embedding-2`. **This is the most expensive notebook of the course: about 550 chat-model
calls**, most of them in the two measured comparisons (sections 9 and 14), plus about 70 one-time calls
to write contextual chunks. On `gpt-oss-20b` that is still well under a dollar, but on a free tier it is
more than a day's quota: read the committed outputs, then re-run the sections you want.

**The notebook keeps its own save files**, as in session 11, so your own `chat.py` tickets are never
touched. The variables are read when `core/config.py` is imported, which is why they are set first.

**Docs:** [Multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent) ·
[Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs) ·
[MCP in LangChain](https://docs.langchain.com/oss/python/langchain/mcp) ·
[MCP specification](https://modelcontextprotocol.io/specification/2026-07-28)

In [22]:
import json
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", category=LangChainBetaWarning)
warnings.filterwarnings("ignore", message="IProgress not found")    # fastembed's progress bar

HERE = Path.cwd()
STATE = HERE.parent / ".state"                   # labs/.state, gitignored
NOTEBOOK_FILES = {"SHOPWISE_CHECKPOINT_DB": STATE / "s12-notebook-checkpoints.sqlite",
                  "SHOPWISE_MEMORY_DB": STATE / "s12-notebook-memory.sqlite",
                  "SHOPWISE_REFUND_LEDGER": STATE / "s12-notebook-refunds.jsonl"}
for name, path in NOTEBOOK_FILES.items():
    for leftover in (path, Path(f"{path}-wal"), Path(f"{path}-shm")):
        leftover.unlink(missing_ok=True)         # a clean start, every run
    os.environ[name] = str(path)                 # before core/config.py is imported

PROJECT = HERE / "project"
sys.path.insert(0, str(PROJECT))                 # the v9 snapshot beside this notebook

from core.config import settings                 # noqa: E402  (v9: config lives in core/)
from kb import store                             # noqa: E402


def collections() -> dict[str, int]:
    """Every knowledge-base collection v9 can use, with its point count (0 = not built)."""
    client = store.shared_client()
    names = [settings.collection, settings.tickets_collection,
             settings.collection_for("contextual"), settings.collection_for("parent")]
    return {n: client.count(n).count if client.collection_exists(n) else 0 for n in names}


def run_py(*args, timeout=1800):
    """Run a project script in its own process and print what it printed.
    Embedded Qdrant allows one process per folder, so the notebook lets go first."""
    store.close_shared()
    out = subprocess.run([sys.executable, *args], cwd=PROJECT, capture_output=True,
                         text=True, timeout=timeout)
    print((out.stdout.strip() or out.stderr[-2000:]))


if not collections()[settings.tickets_collection]:
    run_py("kb/ingest.py", "--tickets")          # 27 past tickets: embeddings only, no chat calls

print(f"{settings.APP_NAME} v{settings.VERSION} · {settings.model_id} · routing {settings.ROUTING}")
for name, points in collections().items():
    print(f"  {name:48} {points:>4} points")
print(f"MCP transport: {settings.MCP_TRANSPORT} · a chat message counts as received on {settings.today}")
assert collections()[settings.collection], "Empty knowledge base: run `uv run python kb/ingest.py` from project/."

ShopWise Support Assistant v9.0.0 · openrouter:openai/gpt-oss-20b · routing hybrid
  policies__gemini-embedding-2-768                   70 points
  tickets__gemini-embedding-2-768                    27 points
  policies-contextual__gemini-embedding-2-768         0 points
  policies-parent__gemini-embedding-2-768             0 points
MCP transport: stdio · a chat message counts as received on 2026-08-09


The two empty collections are built in section 14, where they are the lesson. Four helpers: `show`
prints one definition from a project file (the code that runs, not a copy), `v8_chat` runs session 11's
`chat.py` in its own process (two versions of the same modules cannot share one kernel), `TICKETS` holds
every ticket both inboxes know, and `turn` runs one message through the team and prints it the way
`chat.py` does.

In [14]:
from chat import render                           # noqa: E402
from inbox import load_all_tickets                # noqa: E402
from langgraph.types import Command               # noqa: E402
from middleware import context_for                # noqa: E402
from workflows.events import paused_on, run_config, stream_ticket   # noqa: E402
from workflows.supervisor import open_team        # noqa: E402

TICKETS = load_all_tickets()
_team = open_team()
graph = _team.__enter__()                          # v9's graph, held open for the whole notebook


def show(name, file):
    """Print one top-level definition from a project file: the real code, not a copy."""
    lines = (PROJECT / file).read_text().splitlines()
    start = next(i for i, l in enumerate(lines)
                 if l.startswith((f"def {name}(", f"{name} = ", f"class {name}(", f"class {name}:")))
    end = start + 1
    while end < len(lines) and (not lines[end] or lines[end][0] in " )]}"):
        end += 1
    while start > 0 and lines[start - 1].startswith("@"):
        start -= 1
    print("\n".join(lines[start:end]).rstrip())


def v8_chat(email, lines):
    """Run session 11's chat.py (v8) in its own process, typing `lines` into it."""
    store.close_shared()
    out = subprocess.run([sys.executable, "chat.py", "--as", email],
                         cwd=HERE.parent / "session-11" / "project",
                         input="\n".join(lines) + "\n", capture_output=True, text=True, timeout=600)
    print(out.stdout.strip() or out.stderr[-1500:])


def turn(thread, text, email, role="customer"):
    """One chat line through the team, printed as chat.py prints it."""
    render(stream_ticket(graph, thread, text, context_for(email, role=role)))


def route_of(thread):
    """The turn's route_log, one line per decision."""
    for e in graph.get_state(run_config(thread)).values.get("route_log", []):
        print(f"  {e['by']:10} → {e['to']:9} {e.get('reason', '')}")

## 2. The failures: one overloaded employee, one missing source

Two openings, one per part. Each is a problem you will watch happen.

**Failure A: one overloaded employee.** Before the inbox graph, ShopWise's chat ran on one agent, and its
instructions show what happens when one employee owns everything. Here are v8's chat-agent instructions,
headings only, and two lines that pull against each other:

In [15]:
import yaml                                         # noqa: E402

v8_prompt = yaml.safe_load((HERE.parent / "session-11/project/prompts/chat.yml").read_text())["agent_prompt"]
print(f"v8's chat agent: {len(v8_prompt.split())} words of instructions, 8 tools\n")
print("\n".join(l for l in v8_prompt.splitlines() if l.startswith("## ")))
print()
for needle in ("store in Bangladesh", "two to five short sentences", "After each sentence that states"):
    print("…", next(l.strip() for l in v8_prompt.splitlines() if needle in l), "…")

v8's chat agent: 541 words of instructions, 8 tools

## Identity
## Instructions
## Constraints, and why they exist
## Refunds
## When a lookup finds nothing
## Policy questions
## Output format

… store in Bangladesh selling audio, wearables and small kitchen appliances. …
… Plain text, two to five short sentences. No markdown; the square-bracket …
… - Answer only from the passages it returns. After each sentence that states …


Read the last three lines together. The identity says a store **in Bangladesh**; the shipping policy it
must cite says ShopWise ships **within the United States only**. And a reply must be "two to five short
sentences" while also putting a citation after **every** sentence that states a rule. On a four-question
ticket, something has to give. Every new responsibility added to one prompt is a new chance for two rules
to collide, and nobody owns the collision.

The inbox graph (v7–v8) answered this by splitting work across desks, routed **by code**. Now give it
Kamal's ticket, where one half of the question depends on the other:

In [17]:
v8_chat("kamal@example.com", ["Is your BrewMaster kettle overpriced, and can I return it if it is?", "/quit"])

ShopWise Support Assistant v8.0.0 · openrouter:openai/gpt-oss-20b · logged in as kamal@example.com · ticket CHAT-00E735
commands: /new /thread /history /quit · manager: /approve /edit /reject

customer  > Is your BrewMaster kettle overpriced, and can I return it if it is?
assistant > · Searching the policies… · checking get_product… · Found 4 policy passages Hi Kamal,

The BrewMaster Electric Kettle (SKU‑300) is priced at $49.00 and comes with a 6‑month warranty.  

Our policy documents do not cover refunds or returns for items considered overpriced. A member of the ShopWise team will follow up.

customer  > /quit
Ticket CHAT-00E735 is saved. Reopen it with:  uv run python chat.py --as kamal@example.com --thread CHAT-00E735


**What v8 did.** It searched the policies and checked `get_product`, gave the price ($49.00) and the
warranty, then said the policies "do not cover refunds or returns for items considered overpriced" and
handed off to a person. The two halves were never connected: nobody judged whether the kettle *is*
overpriced, and the return question was read as "can I return an overpriced item?" instead of "if it is,
can I return it?". The dependency between the two halves had no owner.

*"We don't need a bigger brain. We need a team, and a manager who reads what each member found and
decides what happens next."* Then the honesty slide, which this whole part keeps returning to: **most
tasks do not need a team.** LangChain's own benchmark (section 4) shows a team costs more calls on a
simple question.

**Failure B: the question the policies cannot answer.** Tanvir writes: *"My PulseFit strap snapped after
two months. Has this happened to other customers, and what did you do?"* The policy collection holds the
warranty rule, and no history at all. v8's policy graph:

In [24]:
from kb import rag                                                     # noqa: E402
from workflows.corrective_rag import build_corrective                  # noqa: E402

v8_policy = build_corrective(grade=False, fusion=False, hyde=False)    # v8's policy desk, straight
strap = "My PulseFit strap snapped after two months. Has this happened to other customers, and what did you do?"
print(v8_policy.invoke({"question": strap})["answer"])

Because your PulseFit strap is a wearables product, it is covered by our 24‑month warranty [warranty-policy · 2. How long each product type is covered]. To make a claim, send support your order number and a photo showing the fault; we’ll send a replacement if it’s in stock or refund it otherwise, and cover the return shipping [warranty-policy · 3. Making a claim]. The warranty team will review your claim and reply within two working days [warranty-policy · 3. Making a claim].


**What v8 answered.** The rule, well cited: 24 months for wearables, and how to make a claim. The other half,
*"has this happened to other customers, and what did you do?"*, it skipped without saying so. The policies
hold no history, so there was nothing to answer it from, and the reply reads as complete anyway.

And a second, quieter failure in the same desk. Every chunk has carried `authority` ("shop" or
"regulator") since session 8, and nothing uses it. Ask what the **law** says:

In [6]:
for q in ["What does the regulator say about how long a seller can take to ship an order?",
          "What's the warranty on wearables?"]:
    print(q)
    for d in rag.search(q):
        print(f"   {rag.label(d):58} authority: {d.metadata['authority']}")
    print()

What does the regulator say about how long a seller can take to ship an order?
   [ftc-prompt-delivery · p.1]                                authority: regulator
   [ftc-prompt-delivery · p.1]                                authority: regulator
   [ftc-prompt-delivery · p.1]                                authority: regulator
   [ftc-prompt-delivery · p.1]                                authority: regulator

What's the warranty on wearables?
   [warranty-policy · 2. How long each product type is covered] authority: shop
   [warranty-policy · 1. What the warranty covers]            authority: shop
   [warranty-policy · preamble]                               authority: shop
   [warranty-policy · 3. Making a claim]                      authority: shop



**What ranked where.** On this run the regulator question got lucky: all four passages are the FTC rule,
because the question is worded much like the rule itself. The wearables question returned the shop's
warranty sections and nothing else. Neither result used `authority`; the order came from similarity alone.
Word the regulator question less like the regulation ("how long can you take to ship?") and the shop's
shipping policy would rank first, because nothing checks whose rule a passage is.
The metadata that would settle it is right there in each passage, unread.

## 3. Where we are on the map

| | v8 (session 11) | v9 (tonight) |
|---|---|---|
| Who routes | code: one desk per issue, by category | code for one clear issue; a **supervisor** for the rest |
| Routing visible? | in the graph's edges | in **state**: `route_log`, readable in every trace |
| A conversation that changes subject | answered by whichever desk got it | **handed off** with a three-line summary |
| Knowledge | policies only | policies **and** past tickets, cited differently |
| Metadata | ignored | **self-query** filters, with a no-filter fallback |
| Chunks | one scheme | plain, **contextual** and **parent**, each measured |
| Tools | importable in this process only | served over **MCP** to Claude Code and to our own agent (`session-12-mcp.ipynb`) |
| Layout | `config.py` among the features | **`core/`**: config, constants, exceptions |

---
# Part A: Multi-agent systems, honestly

## 4. When not to, and the five patterns

**The default answer is no.** People reach for "multi-agent" for three real reasons, and LangChain's docs
name the same three:

| Real reason | ShopWise version |
|---|---|
| **Context management**: one prompt cannot hold every specialist's knowledge well | the overloaded prompt of section 2 |
| **Distributed development**: teams own their parts independently | the policy team tunes retrieval without touching billing's code |
| **Parallelization**: independent subtasks run at once | four issues in one ticket |

If none applies, one agent with good tools is cheaper, faster and easier to debug. If one does, pick the
pattern by what the user needs, using one rule each:

<img src="figures/team-patterns.png" width="860"
     alt="The five multi-agent patterns, each a small diagram with a one-line rule. Subagents: a main
     agent calls workers as tools, and the worker never talks to the user. Handoffs: the active agent
     changes and the user keeps talking to whoever is active. Skills: one agent loads different
     instructions on demand. Router: classify once up front, then send. Custom workflow: a graph you
     wrote, when order must be guaranteed, which v7 and v8 already are. A bottom band says pattern
     choice is arithmetic, not taste.">

| Pattern | Use it when | ShopWise tonight |
|---|---|---|
| **Subagents** | the worker never talks to the user | pattern 1, section 5 |
| **Handoffs** | the user keeps talking, and who they talk to changes | orders → refunds, section 8 |
| **Skills** | one agent needs different *instructions*, not different tools | `load_skill`, section 10 |
| **Router** | you can classify up front | the code route: v7's `DESK_FOR`, kept |
| **Custom workflow** | order must be guaranteed | **v7 and v8 already are one** |

**Pattern choice is arithmetic, not taste.** LangChain's docs measured the patterns on three request
shapes. Model calls per request:

| Request | Subagents | Handoffs | Skills | Router |
|---|:---:|:---:|:---:|:---:|
| one-shot ("buy coffee") | 4 | **3** | **3** | **3** |
| the same request again, same conversation (both turns) | 8 | **5** | **5** | 6 |
| multi-domain ("compare Python, JavaScript and Rust"), calls · tokens | **5 · ~9K** | 7+ · ~14K+ | 3 · ~15K | **5 · ~9K** |

Subagents pay one extra call on a simple request, because the answer flows back through the main agent:
that is the price of central control. Skills make the fewest calls but accumulate tokens, because every
loaded playbook stays in the context. Nothing is free; the table says what each thing costs.

**We are not leaving LangGraph.** The bridge, before anyone asks:

<img src="figures/two-altitudes.png" width="860"
     alt="The two altitudes of orchestration on one spectrum. On the left, orchestration by graph, as in
     v7 and v8: code chooses the route, routing costs zero model calls, the route is static and
     auditable. On the right, orchestration by model, new in v9: a supervisor chooses, it is dynamic and
     handles several domains, and each routing decision costs one call. Underneath both, one shared
     foundation labelled LangGraph. Between them, the shipped choice: hybrid, code for one clear issue
     and the supervisor for the rest.">

v7 and v8 orchestrate **by graph**: code picks the route, routing costs zero model calls, and the result
is static and auditable. Tonight adds orchestration **by model**: dynamic and multi-domain, at the price
of an extra hop. Both run on LangGraph, and ShopWise will ship a mix of the two.

## 5. Pattern 1: subagents as tools

**Count the new primitives: zero.** A specialist is a `create_agent` (session 5). Wrapped in a `@tool`
(session 5), it becomes something another agent can call. The supervisor is a `create_agent` whose only
tools are those wrappers. Here is the policy expert's wrapper, from `workflows/subagents.py`:

In [18]:
show("ask_policy_expert", "workflows/subagents.py")

@tool
def ask_policy_expert(question: str) -> str:
    """Consult the policy specialist for ANY question about rules: returns,
    refunds, shipping, warranty, what the law requires, and how similar cases
    were handled before. Never answer a policy question yourself. Its answer
    carries [label] citations: keep them exactly."""
    filters = extract_filters(question) if settings.SELF_QUERY else NO_FILTERS
    return research_question(question, filters)["answer"]


**The wrapper's docstring is the routing logic.** The supervisor model never sees the specialist's code
or prompt. It sees four tool names and four docstrings, and from those alone it decides whom to consult.
That is session 5's docstring lesson promoted to org design, and exercise 1 breaks it on purpose.

Each wrapper is a few lines because each reuses something built earlier: `ask_orders_expert` runs the
orders specialist (session 7's desk), `ask_policy_expert` the knowledge specialist (section 12), and
`ask_refunds_expert` session 11's `refund_case`. The customer reaches them as **context**, through
`ToolRuntime`, never through the conversation. Tanvir asks two things at once:

In [19]:
from workflows.subagents import answer_with_team   # noqa: E402

out = answer_with_team("Where is my watch (ORD-5003), and how long is its warranty?",
                       context_for("tanvir@example.com"))
print("consulted:", " → ".join(out["consulted"]), "\n")
print(out["reply"])

consulted: ask_orders_expert 

Hi Tanvir, your watch (ORD‑5003) was shipped on July 28 2026 via SwiftEx (tracking SW‑88231). It’s currently in transit and should arrive by August 8, 2026. The product comes with a 24‑month warranty. Let me know if you need anything else!


**Who it consulted.** Only the orders expert. That expert answered the warranty too, from the product
record, so the knowledge expert was never asked and the reply carries no policy citation. One agent
decided that on its own, and nothing outside its messages records why.

Notice where the routing lives: in `out["consulted"]`, which we had to dig out of the agent's tool-call
messages. Nothing in the agent's state says *why* it consulted whom, nothing caps how many times it may,
and a refund's pause would happen inside a tool inside an agent. That is why ShopWise ships the next
pattern instead.

## 6. Pattern 2: the supervisor as a graph

The same team, with the manager as a **node** in a `StateGraph`. The supervisor node makes one structured
call that returns a `Route` (who next, on what task, and why), and returns
`Command(goto=specialist, update=...)`. Each specialist reports back to the supervisor, which reads what
came back and decides again, until it says `FINISH`.

<img src="figures/supervisor-loop.png" width="860"
     alt="The supervisor graph as a loop. The supervisor node in the centre returns Command goto with an
     update. Arrows go out to four specialists: orders, knowledge, refunds and, marked ADV, market; each
     specialist arrow returns to the supervisor. A FINISH arrow leaves to compose and then the reply.
     Beside the supervisor, a route_log card shows three entries with who chose and why, and a hop
     counter showing hops 2 of 4. A side lane shows the code route: one clear issue goes straight to its
     specialist with no model call.">

The `Route` schema is the supervisor's whole vocabulary. `next` is a closed list, so it cannot invent a
desk; `task` is what the specialist sees **instead of** the conversation:

In [5]:
show("Route", "schemas.py")

class Route(BaseModel):
    """One step of the supervisor: who works next, on what, and why.

    `next` is a closed list, so the supervisor cannot invent a desk. `task` is
    what the specialist sees INSTEAD of the conversation: a specialist with a
    clean context answers one question well. `reason` is for people: it is
    written into route_log, where a trace, a reviewer and a test can read it."""

    next: Literal["orders", "knowledge", "refunds", "market", "FINISH"] = Field(
        description="The specialist who works next, or FINISH when every part of the "
                    "customer's message has been handled.")
    task: str = Field(
        default="",
        description="One self-contained task for that specialist, with every order id, "
                    "refund id, product and amount it needs. Empty when next is FINISH.")
    order_id: str | None = Field(
        default=None, description="The order the task is about, if it is about one.")
    reason: str = Fiel

The node, in four cases, cheapest first. Read case 2 carefully: it is v7's router, kept.

In [6]:
show("supervisor", "workflows/supervisor/nodes.py")

def supervisor(state: TeamState, routing: str | None = None) -> Command:
    routing = routing or settings.ROUTING
    log = state.get("route_log") or []
    subs = state.get("sub_issues") or []

    # 1. a code-routed turn is finished once its specialists report back
    if any(entry["by"] == "code" for entry in log):
        return finish("every issue went to its specialist by category", by="code")

    # 2. the code route: no model call
    clear = len(subs) == 1 and subs[0].category in CLEAR_CATEGORIES
    if not log and subs and (routing == "code" or (routing == "hybrid" and clear)):
        sends, entries = [], []
        for i, sub in enumerate(subs):
            target = SPECIALIST_FOR[sub.category]
            sends.append(Send(target, {"task": task_for(sub.question, i, sub.order_id, sub.category),
                                       "ticket": state["ticket"]}))
            entries.append({"by": "code", "to": target, "task": sub.question,
                            "reason

| Case | When | Model calls |
|---|---|---|
| 1. finish a code-routed turn | its specialists have reported | 0 |
| 2. **the code route** | `split` found **one** issue with a clear category (`ROUTING=hybrid`) | 0 |
| 3. the hop cap | `MAX_HOPS` specialists already called this turn | 0 |
| 4. the model decides | everything else | 1 per decision |

**Why a hop cap in state?** The supervisor and its specialists form a cycle, and session 10's rule holds:
a cycle without an exit is a bill without a ceiling. `hops` lives in the state, so the cap holds even if
the model never says `FINISH`.

**`Command` does the routing and the update in one return value.** There are no conditional edges out of
the supervisor; `destinations=` in `graph.py` exists only so the drawing shows the arrows.

A single clear question first. It should take the code route, with no supervisor call at all:

In [7]:
turn("S12-ONE", "How long is the warranty on a smartwatch?", "tanvir@example.com")
route_of("S12-ONE")

assistant > · Routed by category: knowledge Hi Tanvir,

The smartwatch falls under the “wearables” category, which carries a 24‑month warranty from the delivery date. This period covers faults in materials or workmanship that appear in normal use【warranty-policy · 2. How long each product type is covered】.
  code       → knowledge category warranty
  code       → FINISH    every issue went to its specialist by category


**The code route.** Triage saw one clear issue (category `warranty`), so code sent it to the knowledge
specialist and then to `FINISH`: no supervisor call at all, and the reply cites the warranty policy.

Now Kamal's ticket from section 2, where
the second half depends on the first:

In [8]:
turn("S12-KAMAL", "Is your BrewMaster kettle overpriced, and can I return it if it is?", "kamal@example.com")
route_of("S12-KAMAL")

assistant > · Routed by category: orders · checking get_product… Hi Kamal, thank you for reaching out. The BrewMaster Electric Kettle (SKU‑300) is priced at $49.00 and, as of now, it is not in stock. It comes with a six‑month warranty. At this time we don’t have specific return or refund details for this product, so please review our general return policy for eligibility guidance.
  code       → orders    category product_question
  code       → FINISH    every issue went to its specialist by category


**What happened on this run.** Triage classified Kamal's ticket as a single `product_question`, so the code
route sent it to orders alone and the supervisor never ran. The reply gives the price and stock, then
admits it has no return details: the "can I return it" half went nowhere. And "overpriced" is not
something any core specialist can judge, since it needs market prices (the [ADV] Market Analyst). This is
the weak spot of the code route: a ticket whose second half depends on the first can look like one issue
to triage. Read `route_log` above: it says exactly why.

**Routing is now visible state.** `route_log` is in every checkpoint, so it is in every trace, a
reviewer can read it, and a test with a scripted model can assert on it. In the MCP notebook
(`session-12-mcp.ipynb`, section 5) a manager asks Claude Code about a past routing decision, and the
answer comes from this list.

## 7. Specialists as subgraphs: two ways to compose

A specialist with its own steps is a graph of its own. LangGraph offers two ways to put a graph inside
another, and the rule for choosing is about **state keys**:

<img src="figures/subgraphs-two-ways.png" width="860"
     alt="Two ways to compose a subgraph, side by side. Left, a subgraph added as a node: the knowledge
     graph shares the keys task and findings with the parent, and its private keys filters, documents and
     answer stay inside. Right, a subgraph called inside a node: the refund graph has its own
     RefundState, and a wrapper node translates the parent state in and the result out. Underneath, one
     checkpointer covers parent and children, so the refund's interrupt pauses the whole run and resumes
     through the boundary.">

| | Add it as a node | Call it inside a node |
|---|---|---|
| When | the child **shares keys** with the parent | the child has a **different schema** |
| How | `add_node("knowledge", knowledge_graph())` | a plain node runs `refund_graph.invoke(...)` and translates |
| ShopWise | the knowledge specialist: `task` in, `findings` out | the refunds specialist: `RefundState` in, a finding out |

**The knowledge specialist, added as a node.** Its `input_schema` and `output_schema` make the boundary
exact: only `task` goes in and only `findings` comes out. Its other keys stay private to it.

In [ ]:
show("knowledge_graph", "workflows/specialists/knowledge.py")
from workflows.specialists.knowledge import KnowledgeInput, KnowledgeOutput, KnowledgeState   # noqa: E402
print("\nin:", list(KnowledgeInput.__annotations__), "· out:", list(KnowledgeOutput.__annotations__),
      "· private:", [k for k in KnowledgeState.__annotations__ if k not in ("task", "findings")])

`output_schema` is doing more work than it looks. `findings` **appends**. Without an output schema the
subgraph would hand back its whole state, and any findings it had received would be appended to the
parent a second time. With `input_schema = {task}`, the child's findings start empty and only its own
finding goes out.

**The refunds specialist, called inside a node.** v8's refund flow now has its own graph and its own
`RefundState`, so the specialist is an adapter: supervisor state in, refund state for the graph, a finding
back out.

In [ ]:
show("refunds", "workflows/specialists/refunds.py")

**The checkpointer covers both.** A subgraph compiled with no checkpointer of its own saves into its
parent's checkpoints, under the calling node's namespace. That is the property tonight depends on: the
refund's `interrupt()`, two levels down, must pause **this** thread, and a resume on this thread must
reach it. Ayesha's refund, TCK-M05:

In [ ]:
ayesha, manager = context_for("ayesha@example.com"), context_for("ayesha@example.com", role="manager")
render(stream_ticket(graph, "TCK-M05", TICKETS["TCK-M05"], ayesha))
print("\nparent next:", graph.get_state(run_config("TCK-M05")).next)
for task in graph.get_state(run_config("TCK-M05"), subgraphs=True).tasks:
    child = task.state
    print(f"child of {task.name!r}: next {child.next}, proposal {child.values['proposal']['amount']} "
          f"on {child.values['proposal']['order_id']} ({child.values['proposal']['recommendation']})")

The parent is paused **in** `refunds`, and `get_state(config, subgraphs=True)` reaches into the child: it
is waiting at `approval`, with the proposal it built in `assess`. Now the manager answers, on the
parent's thread:

In [ ]:
render(stream_ticket(graph, "TCK-M05", Command(resume={"type": "approve"}), manager))
print("\nrefunds paid this turn:", [(r["refund_id"], r["amount"]) for r in graph.get_state(run_config("TCK-M05")).values["refunds"]])

⟪AFTER-RUN: confirm paid once, the progress line printed again on resume (rule 1 from session 11).⟫

The parent node ran again from its first line, called the refund graph again, and the refund graph
resumed at `approval` instead of starting over. One detail made the child's state visible: the compiled
refund graph is a **module-level object** in `refunds.py`. LangGraph finds a subgraph that a node calls by
looking at what the node's function refers to; hide it behind a cached function and the pause still works,
but `subgraphs=True` shows nothing.

## 8. Handoffs: passing the conversation on

A subagent answers and is gone. Some specialists should instead pass the **conversation** on: Tanvir asks
where his watch is, then says he would rather cancel and have his money back. The orders specialist found
the shipment; the refunds specialist should take it from there.

<img src="figures/handoff-summary.png" width="860"
     alt="A handoff from the orders specialist to the refunds specialist. On the left, the orders
     specialist's context: the brief, a get_shipment call and its JSON, the customer's change of mind. In
     the middle, a tool named transfer_to_refunds returns Command with graph equal to Command.PARENT. On
     the right, the refunds specialist receives only a three-line summary, not the transcript. A contrast
     strip shows the transcript version crossed out as more tokens and a noisier context.">

The mechanism is a tool that returns a `Command` for the **parent** graph:

In [ ]:
show("transfer_to_refunds", "workflows/supervisor/handoffs.py")

`graph=Command.PARENT` means "not this agent's graph, the one around it". LangGraph stops the orders agent
where it is and applies the Command to the supervisor graph: the summary is written to `handoff`, and
`refunds` runs next.

**Rule: a handoff carries a summary, not the transcript.** It costs fewer tokens, and the receiving
specialist's context stays clean: it never sees the tool calls, the shipment JSON or the small talk.
LangChain's handoff docs also add a `ToolMessage` to the update, because there the agents share one
`messages` list, and an AI tool call left unanswered breaks the next model call. ShopWise's `messages`
hold only the customer's turns and the replies, never tool calls, so there is nothing to answer.

**One trap we hit building this.** The orders node first had a normal edge back to the supervisor. A node
with an edge **and** a `Command` goes to **both**: the supervisor finished the turn while refunds was still
asking a manager. So `orders` routes itself with `Command` and has no static edge (`ROUTES_ITSELF` in
`graph.py`).

In [ ]:
from workflows.supervisor import build_team       # noqa: E402

# The same files, routing="supervisor": a one-issue message must not take the
# code route here, or the orders specialist would never be the first stop.
sup = build_team(checkpointer=graph.checkpointer, store=graph.store, routing="supervisor")
tanvir, tanvir_mgr = context_for("tanvir@example.com"), context_for("tanvir@example.com", role="manager")
render(stream_ticket(sup, "S12-HANDOFF", "Is my PulseFit watch, ORD-5003, still stuck in transit? "
                     "If it is, I'd rather cancel it and get my money back.", tanvir))
route_of("S12-HANDOFF")
print("\nhandoff summary:\n ", (graph.get_state(run_config("S12-HANDOFF")).values.get("handoff") or {}).get("summary", "(none)"))

⟪AFTER-RUN: who routed what, whether the handoff fired, what the summary said, and that it paused (not delivered → recommendation reject).⟫
The cell used a graph built with `routing="supervisor"` on the notebook's own files: with the shipped
hybrid routing, `split` might file this message as one refund issue and send it straight to refunds by
code, which is cheaper and also correct, but shows no handoff. The manager decides it as a cancellation:

In [ ]:
render(stream_ticket(sup, "S12-HANDOFF",
                     Command(resume={"type": "reject", "reason": "it has not been delivered yet: we will cancel the order instead"}),
                     tanvir_mgr))
route_of("S12-HANDOFF")

**Subagents or handoffs, in one sentence:** with subagents the main agent keeps the conversation and the
worker never talks to the customer; with handoffs the customer keeps talking, and the specialist they talk
to changes. The appendix (section A) builds the full returns state machine, the most-requested handoff
pattern in production.

## 9. The measured comparison

Two multi-agent shapes now exist, plus v8's static router. Which should ShopWise ship? Not by taste: the
same eight tickets (four single-issue, four multi-issue, none asking for money) go through four systems,
and the reply is judged the same way for all of them (`evals/team_tickets.yml`: required facts and
required citations).

| System | Routing | Routing calls |
|---|---|---|
| v8 static router | `split`, then one desk per issue by category, in parallel | 0 |
| subagents | the team agent, inside its own loop | one per step of its loop |
| supervisor | the supervisor node, every turn | one per hop |
| **hybrid (shipped)** | code for one clear issue, the supervisor for the rest | 0 or one per hop |

This cell runs `evals/multiagent_compare.py`, which writes `docs/multiagent-comparison.md`. About 200
calls; it takes several minutes.

In [ ]:
run_py("evals/multiagent_compare.py", timeout=3600)

⟪AFTER-RUN: read the two summary tables honestly: single-issue winner on calls/latency, multi-issue correctness; justify ROUTING=hybrid or say what the numbers chose.⟫

This is the result to expect, and the one to teach even when it stings: **on a single, clear question the
static router wins**, because the supervisor's hop is pure overhead there. The supervisor earns its call
only when a ticket spans domains or one part depends on another. So ShopWise keeps both: `split` routes a
single clear issue in code, and everything else goes to the supervisor. `ROUTING` in `core/config.py`
records the decision, and the committed report is its evidence.

## 10. Skills: different instructions, without another agent

When two specialists would differ only in what they are **told**, not in what they can **do**, they are
not two agents. They are one agent with two playbooks. A skill is a playbook the model loads when it
decides it needs one. Until then only a one-line description sits in the tool's docstring: the menu, not
the whole cookbook (progressive disclosure).

In [ ]:
from workflows.skills import SKILLS, load_skill   # noqa: E402

print(load_skill.description, "\n")
print((PROJECT / "skills/gold_tier/SKILL.md").read_text()[:330], "…")

Each skill is a folder with a `SKILL.md`: YAML frontmatter (`name`, `description`) and a markdown body.
That is the open **Agent Skills** format (agentskills.io), the same one Claude Code loads, and session 15
writes real ones for it. The tool's docstring is built from the frontmatter, so adding a skill is adding a
folder. The team agent has `load_skill`; Ayesha's angriest ticket:

In [ ]:
out = answer_with_team(TICKETS["TCK-003"]["body"], context_for("ayesha@example.com"))
print("consulted:", " → ".join(out["consulted"]), "\n")
print(out["reply"])

⟪AFTER-RUN: whether it loaded deescalation (and gold_tier: Ayesha is gold), and how the reply reads.⟫
The benchmark's trade-off, seen here: the fewest calls of any pattern, but each loaded playbook stays in
the context for the rest of the run.

---
# Part B: The knowledge specialist, measured

## 11. The rule, and a longer yardstick

The rule of this part has not changed since session 10: **every technique answers a named symptom, and
ships only if the scorecard moves.** So the yardstick grows first. `evals/rag_eval.yml` adds the questions
v8 cannot get right:

| Kind | Example | What a correct answer needs |
|---|---|---|
| **precedent** (`KBP-`) | "Has a PulseFit strap snapped before, and what did you do?" | a past ticket, cited as a similar case |
| **filtered** (`KBS-`) | "What does the **regulator** say about shipping times?" | the passage the filter implies |

A precedent row counts as **retrieved** when one of its past tickets came back holding the evidence, and
as **cited** when the answer names that ticket *outside* square brackets. Precedent written as a
`[label]` would read as a rule, and that counts as wrong. v8's policy graph on the new rows:

In [9]:
from evals.run_eval import graph_target, rag_eval_rows, score, table   # noqa: E402

RAG_ROWS = rag_eval_rows()
v8_rag = score(graph_target(), RAG_ROWS)
table({"v8 policy graph": v8_rag})
for row in v8_rag["rows"]:
    print(f"  {row['id']}: {'refused' if row['refused'] else 'answered'}"
          f"{'' if not row['answerable'] else ('' if row['retrieved'] else ' · not retrieved')}")

arm              refusal  citation  retrieval  calls/q    s/q
v8 policy graph     0.73      0.71       0.50      1.0    6.3
  KBP-01: refused · not retrieved
  KBP-02: refused · not retrieved
  KBP-03: answered · not retrieved
  KBP-04: refused · not retrieved
  KBP-05: answered · not retrieved
  KBP-06: refused
  KBS-01: answered
  KBS-02: answered
  KBS-03: answered
  KBS-04: answered
  KBS-05: answered


**The red rows.** v8 never retrieved a passage for any precedent row (KBP-01 to KBP-05): the answers are in
past tickets, which v8 cannot search. It refused four, and two (KBP-03, KBP-05) answered without the case
they needed. All five filtered rows (KBS-01 to KBS-05) were answered: similarity alone found a usable
policy for them, so filters will have to earn their extra call on these numbers.

## 12. A second source: past tickets

`kb/ingest.py` now builds a second collection, `tickets__…`, from `kb/past_tickets.yml`: how the team
closed the twenty inbox tickets, plus seven older ones. Each ticket becomes **one** document (subject, what
the customer wrote, how it was resolved), with its category and product type as metadata, and a stable
uuid5 id, so a second ingest overwrites rather than duplicates (session 8's idempotency).

**What is deliberately left out: who wrote.** A past ticket will be shown to *other* customers as
precedent, so `from` never leaves `load_past_tickets()`, and the resolutions say "the customer", never a
name or an order id.

In [20]:
from kb.ingest import load_past_tickets           # noqa: E402

doc = next(d for d in load_past_tickets() if d.metadata["ticket_id"] == "TCK-H01")
print(doc.page_content, "\n")
print({k: v for k, v in doc.metadata.items() if k != "start_index"})

Past ticket TCK-H01 (warranty): watch strap snapped
Customer wrote: The strap on my PulseFit smartwatch snapped at the buckle after about seven weeks of normal wear. I haven't dropped it or pulled on it. Is this covered?
How it was resolved: Treated as a fault in materials under the 24-month wearables warranty: the strap is part of the watch, and it failed in normal use. The customer sent a photo of the buckle; the warranty team approved the claim within two working days and sent a replacement strap, as straps were in stock. ShopWise paid the shipping. 

{'source': 'past-tickets', 'section': 'TCK-H01', 'ticket_id': 'TCK-H01', 'category': 'warranty', 'product_type': 'wearables', 'closed': '2026-04-22', 'authority': 'precedent'}


The knowledge specialist is an agent with **two retriever tools whose docstrings route**: `search_policy`
for rules and `search_past_tickets` for "how did we handle this before". The second docstring carries the
contract's most important rule: **a past ticket is precedent, never policy.**

In [21]:
from workflows.specialists.knowledge import search_past_tickets, search_policy   # noqa: E402

for t in (search_policy, search_past_tickets):
    print(f"{t.name}:\n  {t.description.splitlines()[0]}")
print()
print("\n".join(l for l in (PROJECT / "prompts/team.yml").read_text().splitlines()
                if "PRECEDENT, never a rule" in l or "in a similar case (TCK-H01)" in l))

search_policy:
  Search ShopWise's policy documents and the regulator's documents for RULES. Use it for any question about what is allowed, how long, or what it costs:
search_past_tickets:
  Search how ShopWise's support team resolved similar tickets before: PRECEDENT, not rules. Use it when the question asks whether something has happened before, what

  2. A past ticket is PRECEDENT, never a rule. Say what happened as "in a


The prompt's rule 2 is why the label is different too: `kb/rag.label()` writes a past ticket as `similar
case TCK-H01`, with no brackets. `compose` keeps only `[source · section]` brackets as citations and the
event stream turns those into `citation` events, so a precedent can never be shown to a customer as a
rule. Tanvir's question from section 2:

In [25]:
from workflows.specialists.knowledge import research_question   # noqa: E402

out = research_question(strap)
print(out["answer"], "\n")
print("passages it was shown:", [p.metadata["label"] for p in out["documents"]])

In a similar case (TCK‑H01), a customer reported that the strap on their PulseFit smartwatch snapped after seven weeks of normal use. The support team treated it as a material fault under the 24‑month warranty, approved the claim, and sent a replacement strap, covering all shipping costs. This is the standard response for strap failures that occur during typical use. 

passages it was shown: ['similar case TCK-H01', 'similar case TCK-H07', 'similar case TCK-015', 'similar case TCK-H04']


**What it did.** It searched past tickets: all four passages shown are similar cases (TCK-H01, TCK-H07,
TCK-015, TCK-H04). The answer names the case the way the prompt asks, "in a similar case (TCK-H01)", but
the 24-month rule appears only as part of that case, with no bracketed policy citation. A precedent is not
a rule, so the rule still needs its own citation: a prompt fix, or a reason to send the question to the
policy tool as well.

The filters do not reach the tools as arguments the model chooses. They arrive as **context**
(`ToolRuntime[Lens]`), the way the customer id reaches the authorize rail: the model picks *what* to
search and *where*; code already decided the filters. That is the next section.

## 13. Self-query: the filters a question implies

"What's the warranty on **wearables**?" implies `product_type=wearables`. "What does the **regulator**
say?" implies `authority=regulator`. One structured call reads them out of the question (`Filters` in
`schemas.py`, every field a closed list), and two small functions turn them into Qdrant payload filters:

In [26]:
from kb.query import extract_filters, policy_filter, tickets_filter   # noqa: E402

for q in ["What does the regulator say about how long a seller can take to ship an order?",
          "What's the warranty on wearables?",
          "Has a smartwatch strap ever snapped for another customer?",
          "How do I start a return?"]:
    f = extract_filters(q)
    print(f"{q}\n   filters {f.model_dump(exclude_none=True) or '{}'}")
    for where, flt in (("policies", policy_filter(f)), ("tickets ", tickets_filter(f))):
        conds = [(c.key, getattr(c.match, "value", None) or c.match.any) for c in flt.must] if flt else "no filter"
        print(f"   {where}: {conds}")
    print()

What does the regulator say about how long a seller can take to ship an order?
   filters {'topic': 'shipping', 'authority': 'regulator'}
   policies: [('metadata.source', ['shipping-policy', 'ftc-prompt-delivery']), ('metadata.authority', 'regulator')]
   tickets : no filter

What's the warranty on wearables?
   filters {'topic': 'warranty', 'authority': 'shop', 'product_type': 'wearables'}
   policies: [('metadata.source', ['warranty-policy', 'cfr-16-part-700']), ('metadata.authority', 'shop')]
   tickets : [('metadata.product_type', 'wearables')]

Has a smartwatch strap ever snapped for another customer?
   filters {'topic': 'warranty', 'authority': 'shop', 'product_type': 'wearables'}
   policies: [('metadata.source', ['warranty-policy', 'cfr-16-part-700']), ('metadata.authority', 'shop')]
   tickets : [('metadata.product_type', 'wearables')]

How do I start a return?
   filters {'topic': 'returns', 'authority': 'shop'}
   policies: [('metadata.source', 'returns-policy'), ('metadat

**Which filters came out right.** The regulator question gets `authority: regulator`; the return question
gets `topic: returns`; both strap and wearables questions get `product_type: wearables` for past tickets.
**One over-eager filter:** the wearables and strap questions also get `authority: shop`, although neither
asks whose rule. `topic: warranty` picks two files, the shop's warranty policy and the federal warranty
law (`cfr-16-part-700`), and `authority: shop` then hides the law. Harmless here, but it is exactly the
kind of wrong filter the guardrail below exists for.

Each filter applies only where its metadata exists: `topic` picks the policy files on that subject (the
shop's and the regulator's), `authority` picks shop or law, and `product_type` filters past tickets only,
because policies carry no product type.

**The guardrail: a wrong filter is worse than none.** It hides the right passage completely, and nothing
downstream can bring it back. So three things fall back to **no filter**, never to an empty result:

| Failure | What happens |
|---|---|
| the model returns a value outside a closed list | validation fails, `extract_filters` returns no filters |
| the call itself fails | the same |
| a filtered search returns nothing | `filtered()` searches again without the filter |

The same regulator question as section 2, filtered:

In [ ]:
q = "What does the regulator say about how long a seller can take to ship an order?"
f = extract_filters(q)
for name, flt in (("unfiltered", None), ("filtered", policy_filter(f))):
    print(name, [rag.label(d) for d in rag.search(q, flt=flt)])

`SelfQueryRetriever`, which you will meet in older tutorials, does this with a query-constructor chain. It
now lives in `langchain_classic`; here it is one structured call and two dictionaries, so every step is
visible. Now the scorecard: v8 against the knowledge specialist without and with self-query.

In [ ]:
from evals.run_eval import knowledge_target, score_with_filters   # noqa: E402

rag_results = {"v8 policy graph": v8_rag,
               "knowledge · no filters": score_with_filters(knowledge_target(False), RAG_ROWS),
               "knowledge · self-query": score_with_filters(knowledge_target(True), RAG_ROWS)}
table(rag_results)
for row in rag_results["knowledge · self-query"]["rows"]:
    ok = (row["refused"] if not row["answerable"] else row["retrieved"] and row["cited_right"])
    print(f"  {'  ' if ok else '✗ '}{row['id']}: filters {row['filters'] or '{}'}   (implies {row['implies'] or '{}'})")

⟪AFTER-RUN: read the three columns: what the second source bought (precedent rows), what self-query added or did not on top, and the cost in calls/q. Decide shipped/off.⟫

## 14. Ingest-time fixes: when a chunk has lost its context

Two techniques that change the **collection**, not the question, and so cost nothing per question.

<img src="figures/ingest-fixes.png" width="860"
     alt="Two ingest-time fixes for a chunk that lost its context. Left, contextual retrieval: one model
     call per chunk writes a short sentence situating the chunk in its document, and the sentence is
     prepended before embedding. Right, parent-document retrieval: small child chunks are matched
     precisely, and the whole parent section is returned. Each fix is a new collection, so there are
     three policy collections side by side: plain, contextual and parent.">

| | Contextual chunks | Parent document (small-to-big) |
|---|---|---|
| Symptom | a chunk never says which policy it is from | the matched chunk is too small to answer from |
| Fix | one model call per chunk writes 50–100 tokens situating it, prepended **before embedding** | index small children; return the whole parent section |
| Cost | ~70 calls, **once**, at ingest; cached in `kb/contexts.json` | more tokens per answer (whole sections) |
| In `langchain_classic` | — | `ParentDocumentRetriever` (a lookup table here) |
| Collection | `policies-contextual__…` | `policies-parent__…` |

Anthropic's measurements on its own corpora: contextual embeddings cut top-20 retrieval failures from 5.7%
to 3.7% (−35%); with contextual BM25, to 2.9% (−49%); with reranking too, to 1.9% (−67%). Session 9's
hybrid search is the BM25 half. **Those are their numbers on their corpus.** Ours is 70 chunks, and only
the scorecard can say whether any of it helps here.

**One collection per scheme** (ADR-0010): a collection is one embedding space *and* one chunking scheme. A
contextual chunk and a plain chunk of the same clause are different vectors; in one collection, every
search would return the clause twice. Build both (the contextual build calls the model once per chunk, the
first time only):

In [ ]:
run_py("kb/ingest.py", "--scheme", "contextual")
run_py("kb/ingest.py", "--scheme", "parent")

What a contextual chunk looks like: the model's sentence, then the original text, embedded together.

In [ ]:
contexts = json.loads((PROJECT / "kb/contexts.json").read_text())
from kb.ingest import chunk, chunk_id, load_corpus   # noqa: E402
c = next(c for c in chunk(load_corpus()) if "Wearables" in c.page_content)
print("CONTEXT:", contexts[chunk_id(c)], "\n")
print("CHUNK:  ", " ".join(c.page_content.split())[:300], "…")

One scorecard, three columns: two-step RAG on the gold set plus the filtered questions, changing only
the collection.

In [ ]:
from evals.run_eval import score_schemes   # noqa: E402

scheme_results = score_schemes()
table(scheme_results)
for name, r in scheme_results.items():
    print(f"  {name:10} retrieval misses: {', '.join(x['id'] for x in r['rows'] if x['answerable'] and not x['retrieved']) or 'none'}")

⟪AFTER-RUN: honest verdict per scheme; if nothing moved, say "built but off" and why that is the right answer on a 70-chunk corpus.⟫

## 15. [OPT] Cache what repeats

The same policy question asked fifty times a day does not need fifty searches and fifty answers. LangGraph
caches a node's result by a key you choose:

```python
builder.add_node("research", research, cache_policy=CachePolicy(key_func=cache_key, ttl=3600))
builder.compile(cache=InMemoryCache())
```

The key is the question and its filters, nothing else, so two customers asking the same policy question
share an answer and nothing about either of them is in the key. **Never cache anything customer-specific
or anything with a side effect.** The `note` node after `research` is deliberately *not* cached: it carries
the turn's position, and a cached position would be last week's. Time it:

In [ ]:
from workflows.specialists import knowledge   # noqa: E402

os.environ["SHOPWISE_KB_CACHE_TTL"] = "3600"
settings.KB_CACHE_TTL = 3600
knowledge.knowledge_graph.cache_clear()
cached = knowledge.knowledge_graph()
for attempt in (1, 2):
    started = time.perf_counter()
    cached.invoke({"task": {"question": "How long do I have to return unopened headphones?", "position": 0}})
    print(f"attempt {attempt}: {time.perf_counter() - started:.2f}s")
settings.KB_CACHE_TTL = 0                        # back to the shipped default
os.environ.pop("SHOPWISE_KB_CACHE_TTL")
knowledge.knowledge_graph.cache_clear()

⟪AFTER-RUN: the two timings.⟫ It ships **off** (`KB_CACHE_TTL=0`): an in-process cache dies with the
process and is not shared between servers. Session 15's deployment is where a shared cache (Redis, or the
provider's own prompt cache) would earn a row of its own.

## 16. The decision table

`kb/RAG-DECISIONS.md` was started in session 10 and grew in session 11. Each row is a symptom, the
technique tried against it, what it costs, whether it moved **our** number, and what shipped. It is the
answer to every "should we add X to our RAG?" meeting.

In [ ]:
print((PROJECT / "kb/RAG-DECISIONS.md").read_text().split("## How to read the table")[0])

---
## 17. v9: what you built

**The structural move first.** One move this session, shown as a before and after:

```
before (v8)                          after (v9)
project/                             project/
├── config.py                        ├── core/
│                                    │   ├── config.py       moved, unchanged but for v9's settings
│                                    │   ├── constants.py    the specialists, the code route, the return window
│                                    │   └── exceptions.py   ShopWiseError and four named failures
├── workflows/                       ├── workflows/
│   ├── inbox/                       │   ├── inbox/          v8's graph, kept: the comparison runs it
│   └── corrective_rag/              │   ├── corrective_rag/
│                                    │   ├── supervisor/     state, nodes, graph, handoffs
│                                    │   ├── specialists/    orders, knowledge, refunds, market, mcp_client
│                                    │   └── refund/         v8's refund flow as its own graph
└── ...                              ├── mcp_server/         server, poisoned_demo, audit (the MCP notebook)
                                     ├── skills/             two SKILL.md playbooks
                                     └── .mcp.json           Claude Code's project registration
```

*Why `core/`.* With more than one agent, failures need names. `KeyError: 'refunds_2'` in a trace says a
lookup failed somewhere; `RoutingError` says the supervisor chose a desk that does not exist. A base class,
`ShopWiseError`, lets one `except` tell our deliberate failures from bugs, and session 13 maps each
subclass to an HTTP status. Config, constants and exceptions are what every feature depends on and no
feature owns, so they get their own home. Every import is now `from core.config import settings`.

**Nothing is thrown away.** The desks became specialists; v8's refund flow lives on as the refund graph;
the corrective-RAG graph and session 11's rewrite node are still there; and single clear issues still take
v7's code route, because the comparison said so.

<img src="figures/v9-architecture.png" width="900"
     alt="The v9 architecture. chat.py and inbox.py call stream_ticket, unchanged. The supervisor graph runs
     intake, summarize, rewrite, triage, recall and split, then the supervisor, which takes the code route
     for one clear issue or decides with a model call. It routes to the orders specialist, whose tools
     arrive over MCP; the knowledge specialist, a subgraph searching policies and past tickets with
     self-query filters; and the refunds specialist, which calls the refund graph with its pause for a
     manager. Orders can hand off to refunds. FINISH goes to compose and remember. Below, Qdrant holds four
     collections; on the right, the ShopWise MCP server serves both the orders specialist and Claude Code.">

**The regression: session 11's whole story, through the team.** `chat.py` changed by one import, because
it only ever spoke to `stream_ticket()`. Three separate processes, with the notebook's ledger reset first:

In [ ]:
from shopwise_data import reset_ledger            # noqa: E402
from workflows.specialists import mcp_client      # noqa: E402  (the orders specialist's MCP connection)

reset_ledger()
mcp_client.close()                                # the kernel's MCP server: chat.py starts its own
store.close_shared()


def chat(email, lines, thread=None):
    """Run chat.py in its own process, typing `lines` into it, and print the session."""
    args = [sys.executable, "chat.py", "--as", email] + (["--thread", thread] if thread else [])
    out = subprocess.run(args, cwd=PROJECT, input="\n".join(lines) + "\n",
                         capture_output=True, text=True, timeout=900)
    print(out.stdout.strip() or out.stderr[-2000:])


chat("ayesha@example.com", [
    "Hi, please put my headphones refund through. It's ORD-5001, REF-9001: the left earcup crackles. "
    "And please only ever email me, I can't take calls at work.",
    "/quit"], thread="S12-STORY")

In [ ]:
chat("ayesha@example.com", [
    "/edit 99 only the left earcup is faulty",
    "What's the return window for headphones?",
    "And for the watch?",
    "/quit"], thread="S12-STORY")
ledger = settings.REFUND_LEDGER
print("\nrefund ledger:", ledger.read_text().strip() if ledger.exists() else "(empty: no refund was paid)")

In [ ]:
chat("ayesha@example.com", ["Any news on my two kettles, ORD-5002?", "/quit"])

⟪AFTER-RUN: confirm pause, edit paid once, follow-up rewritten and cited, preference remembered on the new ticket; name anything that regressed.⟫

And session 9's scorecard, re-run on v9's policy graph so a regression in shared code would show:

In [ ]:
run_py("evals/run_eval.py", "--target", "graph")

From `project/`:

```bash
uv run python chat.py --as tanvir@example.com                     # the team, in a chat
uv run python inbox.py TCK-M04 --steps                            # four issues: watch the routing
uv run python inbox.py TCK-M04 --graph v8                         # the same ticket on v8's graph
uv run python evals/multiagent_compare.py                         # the comparison, rewrites docs/
uv run python evals/run_eval.py --rag-eval                        # precedent + filtered questions
uv run python evals/run_eval.py --schemes                         # plain vs contextual vs parent
uv run python kb/ingest.py --scheme contextual                    # build one collection
uv run python -m workflows.supervisor.graph                       # topology, rewrites diagram.md
uv run python -m core.config                                      # settings, and every collection name
```

## 18. Where this leaves us

| v8 | v9 |
|---|---|
| One graph, desks fixed by code | A supervisor and specialists, clean contexts, a visible `route_log` |
| One route per ticket | The supervisor for multi-domain tickets, the code route where the table says it wins |
| — | A pattern vocabulary: five patterns, and when **not** to use any |
| Policies only; metadata ignored | Policies and past tickets, self-query filters, contextual and parent chunks, each measured |

**Limitation.** The team's tools still live inside ShopWise's own process, as far as anyone else can tell.
The ops team works in Claude, and Claude Code cannot read a trace or an order. → `session-12-mcp.ipynb`:
the same tools, served once, to any AI host.

### 18.1 Interview questions

1. When is multi-agent a mistake? Give the cost argument from the benchmark and from your own comparison
   table.
2. Subagents, handoffs and a supervisor graph: one sentence each on when to use it.
3. A RAG bot answers "warranty on wearables?" with the audio clause. What is your first hypothesis, and
   your fix? (Metadata ignored: a self-query filter, with a no-filter fallback.)
4. Your web-search agent quotes a price from a random forum. What do you change? (Source-quality rules and
   a citation requirement in the prompt, not the temperature.)

Exercises: `exercises.ipynb` (exercises 1, 3 and 4 go with this notebook).

In [ ]:
mcp_client.close()                                # the orders specialist's MCP server process
_team.__exit__(None, None, None)                  # the notebook's SQLite connections
store.close_shared()
print("closed")

---
## Appendix: for afterwards

### A. Handoffs, the full treatment: the returns state machine

The most-requested production pattern, adapted from LangChain's customer-support handoff tutorial. One
agent whose **step** changes: each step has its own prompt, its own tools, and preconditions on state. A
tool advances the step by returning `Command(update=...)` with a matching `ToolMessage` (here the
messages *are* shared, so the tool call must be answered):

```python
class ReturnsState(AgentState):
    current_step: NotRequired[Literal["identify", "eligibility", "label"]]
    order_id: NotRequired[str]

STEP_CONFIG = {
    "identify":    {"prompt": IDENTIFY_PROMPT,    "tools": [get_order, record_order],   "requires": []},
    "eligibility": {"prompt": ELIGIBILITY_PROMPT, "tools": [search_policy, approve],    "requires": ["order_id"]},
    "label":       {"prompt": LABEL_PROMPT,       "tools": [send_label],                "requires": ["order_id"]},
}

@tool
def record_order(order_id: str, runtime: ToolRuntime) -> Command:
    """Record the order this return is for, and move on to checking eligibility."""
    return Command(update={"order_id": order_id, "current_step": "eligibility",
                           "messages": [ToolMessage(f"Order {order_id} recorded.",
                                                    tool_call_id=runtime.tool_call_id)]})

@wrap_model_call
def apply_step(request, handler):
    step = request.state.get("current_step", "identify")
    cfg = STEP_CONFIG[step]
    missing = [k for k in cfg["requires"] if not request.state.get(k)]
    if missing:                                   # a precondition failed: back to the start
        cfg = STEP_CONFIG["identify"]
    return handler(request.override(system_prompt=cfg["prompt"], tools=cfg["tools"]))
```

The customer keeps talking to one agent across turns while its job changes under it: that is a handoff,
and the org chart predicts it (the case study in `exercises.ipynb`).

### B. A router with `Send` fan-out across knowledge bases

When sources are known up front, skip the agent: classify which collections a question needs (one
structured call), `Send` one search per collection in parallel, and fuse the results. That is the docs'
`router-knowledge-base` pattern, and session 10's RAG-Fusion with sources in place of phrasings.

### C. GraphRAG and LightRAG

Some questions are multi-hop over entities: *"which gold-tier customers had warranty claims on
wearables?"* Chunks cannot answer that; a knowledge graph of customers, orders, products and claims can.
Microsoft GraphRAG builds community summaries over an extracted graph; LightRAG (`pip install
lightrag-hku`) combines graph and vector retrieval more cheaply. Both re-index with their own model and
dimensions, so neither reuses our collections. RAPTOR (recursive summaries) and late-interaction models
(ColBERT, supported by Qdrant's multivectors) are the other two names you will meet.

### D. The `langchain_classic` retrievers, by name

| Class | What it does | In this course |
|---|---|---|
| `SelfQueryRetriever` | a query-constructor chain turns a question into a filter | `kb/query.extract_filters` |
| `ParentDocumentRetriever` | small chunks indexed, parents returned from a docstore | the `parent` scheme |
| `MultiVectorRetriever` | several vectors (summaries, questions) per document | not built |
| `ContextualCompressionRetriever` | trims retrieved passages to what matters | session 9's reranker is its cousin |
| `EnsembleRetriever` | fuses several retrievers' rankings | session 10's RRF |

In LangChain v1 they moved to `langchain_classic`. The course builds each one as a node or an ingest step,
because each is fifteen visible lines.

### E. Beyond the team: Deep Agents

**Deep Agents** (`deepagents`) packages everything in Part A (planning, a virtual filesystem, subagents,
human-in-the-loop) into one call; session 20 shows it. And LangChain's context-engineering guide has a
fifteen-example matrix of *what* each agent should see, worth keeping open while you design one.
